# WOD-E2E Blind Submission — Colab Runner

Generates `kinematic_constant_velocity.tar.gz` directly from GCS — no local TFRecord download.

**Steps:** authenticate → clone repo → install deps → generate candidates → write tar → download

In [ ]:
# 1. Authenticate with Google Cloud
from google.colab import auth
auth.authenticate_user()
print('Authenticated.')

In [ ]:
# 2. Clone repo
import os

REPO_URL = 'https://github.com/amtellezfernandez/minimal-shot-av'
REPO_DIR = '/content/minimal-shot-av'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())

In [ ]:
# 3. Install protobuf and add paths
import sys

!pip install protobuf --quiet

for p in [f'{REPO_DIR}/.wod-protos', f'{REPO_DIR}/src']:
    if p not in sys.path:
        sys.path.insert(0, p)

# Verify protos and TF (Colab has TF pre-installed)
import tensorflow as tf
from waymo_open_dataset.protos import end_to_end_driving_submission_pb2
print('TF:', tf.__version__, '| Protos OK')

In [ ]:
# 4. Config
GCS_TEST_GLOB    = 'gs://waymo_open_dataset_end_to_end_camera_v_1_0_0/test_202504211836-202504220845.tfrecord-*'
ACCOUNT_NAME     = 'amtellezfernandez@gmail.com'
AUTHORS          = 'Alba Maria Tellez Fernandez'
METHOD_NAME      = 'minimal_shot_av_blind_kinematic_constant_velocity'
DESCRIPTION      = 'Blind kinematic constant-velocity WOD-E2E submission.'
NUM_SHARDS       = 8

CANDIDATES_PATH  = '/content/kinematic_candidates.jsonl'
SUBMISSION_PATH  = '/content/kinematic_constant_velocity.tar.gz'

print('Config OK')

In [ ]:
# 5. Generate kinematic candidates straight from GCS
from minimal_shot_av.model.kinematic_candidates import write_kinematic_candidate_jsonl
from minimal_shot_av.model.wod_e2e import load_preference_frames

frames = load_preference_frames(
    '.',
    shard_glob=GCS_TEST_GLOB,
    include_camera_images=False,
    require_preferences=False,   # test split has no preference labels
)

count = write_kinematic_candidate_jsonl(frames, CANDIDATES_PATH)
print(f'Wrote {count} candidates to {CANDIDATES_PATH}')

In [ ]:
# 6. Write submission tar
from minimal_shot_av.model.wod_submission import (
    WodSubmissionMetadata,
    selected_predictions_from_jsonl,
    write_submission_tar,
)

predictions = selected_predictions_from_jsonl(
    CANDIDATES_PATH,
    candidate_name='constant_velocity',
)

metadata = WodSubmissionMetadata(
    account_name=ACCOUNT_NAME,
    unique_method_name=METHOD_NAME,
    authors=[AUTHORS],
    description=DESCRIPTION,
)

out = write_submission_tar(predictions, SUBMISSION_PATH, metadata, num_shards=NUM_SHARDS)
print(f'Written: {out}  ({len(predictions)} predictions)')

In [ ]:
# 7. Validate
from minimal_shot_av.model.wod_submission import validate_submission_tar
import tarfile, json

report = validate_submission_tar(SUBMISSION_PATH)
print(json.dumps(report, indent=2))

with tarfile.open(SUBMISSION_PATH, 'r:gz') as t:
    print('Tar members:', [m.name for m in t.getmembers() if m.isfile()])

In [ ]:
# 8. Download to your machine
from google.colab import files
files.download(SUBMISSION_PATH)